In [23]:
import os
import numpy as np
import pandas as pd
import librosa
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

import joblib

import os

print(os.listdir(r"C:\Users\nataraj\Downloads\RAVDESS"))

['.ipynb_checkpoints', 'archieve', 'speech_recognition', 'Untitled.ipynb', 'Untitled1.ipynb', 'Untitled2.ipynb']


In [24]:
DATASET_PATH = r"C:\Users\nataraj\Downloads\ravdess\archieve"

print(os.path.exists(DATASET_PATH))
print(os.listdir(DATASET_PATH)[:5])

True
['Actor_01', 'Actor_02', 'Actor_03', 'Actor_04', 'Actor_05']


In [25]:
def extract_features(file_path):

    signal, sr = librosa.load(file_path, sr=None)

    mfcc = librosa.feature.mfcc(
        y=signal,
        sr=sr,
        n_mfcc=40
    )

    mfcc_mean = np.mean(mfcc.T, axis=0)

    return mfcc_mean

In [26]:
sample_file = None

for root, dirs, files in os.walk(DATASET_PATH):
    for file in files:
        if file.endswith(".wav"):
            sample_file = os.path.join(root, file)
            break
    if sample_file:
        break

print(sample_file)

C:\Users\nataraj\Downloads\ravdess\archieve\Actor_01\03-01-01-01-01-01-01.wav


In [27]:
emotion_map = {
    "01": "neutral",
    "02": "calm",
    "03": "happy",
    "04": "sad",
    "05": "angry",
    "06": "fearful",
    "07": "disgust",
    "08": "surprised"
}

In [28]:
def extract_features(file_path):

    signal, sr = librosa.load(file_path, sr=None)

    mfcc = librosa.feature.mfcc(
        y=signal,
        sr=sr,
        n_mfcc=40
    )

    mfcc_mean = np.mean(mfcc.T, axis=0)

    return mfcc_mean

In [29]:
features = []
labels = []

for root, dirs, files in os.walk(DATASET_PATH):

    for file in files:

        if file.endswith(".wav"):

            try:
                file_path = os.path.join(root, file)

                emotion_code = file.split("-")[2]

                emotion = emotion_map[emotion_code]

                feature = extract_features(file_path)

                features.append(feature)
                labels.append(emotion)

            except Exception as e:
                print("Error:", file, e)

print("Total Samples:", len(features))

Total Samples: 1440


In [30]:
df = pd.DataFrame(features)

df["emotion"] = labels

print(df.shape)

df.head()

(1440, 41)


,0,1,2,3,4,5,6,7,8,9,...,31,32,33,34,35,36,37,38,39,emotion
0,-726.217224,68.541420,3.293398,12.205300,5.510278,13.667410,-2.983828,3.098029,-3.310813,-1.564384,...,-1.399110,-2.926856,0.013957,-0.490734,-0.570905,0.040399,-1.207218,-1.594982,-1.436487,neutral
1,-719.128296,70.201569,1.168397,13.122543,7.836950,14.411290,-4.111360,4.468973,-3.539367,-3.658608,...,-2.521470,-2.987673,0.409735,-0.484184,-1.398391,0.255204,-0.984978,-2.093061,-1.040791,neutral
2,-714.995728,69.689346,3.924564,11.924190,6.421723,11.011614,-2.878103,4.509558,-4.476109,-2.671549,...,-0.909152,-3.045955,-0.373294,-0.849145,-0.922105,-0.170320,-1.144423,-1.725612,-1.450561,neutral
3,-710.975281,67.564888,5.782241,13.230727,6.190845,12.628252,-1.675169,5.657494,-4.950634,-3.477545,...,-1.329651,-2.513405,-0.190276,-0.645949,-0.553919,0.459299,-1.580085,-1.647682,-1.509511,neutral
4,-759.921753,75.783524,6.023605,14.557394,6.454188,14.631508,-3.004551,4.620970,-5.200016,-0.707430,...,-2.188582,-2.835501,0.463746,-1.019167,-1.411441,0.350433,-1.519892,-1.250112,-0.613852,calm


In [31]:
df.to_csv("emotion_features.csv", index=False)

print("CSV Saved Successfully")

CSV Saved Successfully


In [32]:
X = df.drop("emotion", axis=1)

y = df["emotion"]

encoder = LabelEncoder()

y = encoder.fit_transform(y)

In [33]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(X_train.shape)
print(X_test.shape)

(1152, 40)
(288, 40)


In [34]:
model = RandomForestClassifier(
    n_estimators=300,
    random_state=42
)

model.fit(X_train, y_train)

print("Training Complete")

Training Complete


In [35]:
pred = model.predict(X_test)

accuracy = accuracy_score(y_test, pred)

print("Accuracy:", round(accuracy*100,2), "%")

Accuracy: 61.11 %


In [36]:
print(
    classification_report(
        y_test,
        pred,
        target_names=encoder.classes_
    )
)

              precision    recall  f1-score   support

       angry       0.78      0.74      0.76        38
        calm       0.63      0.89      0.74        38
     disgust       0.51      0.58      0.54        38
     fearful       0.57      0.77      0.65        39
       happy       0.67      0.36      0.47        39
     neutral       0.50      0.21      0.30        19
         sad       0.56      0.53      0.54        38
   surprised       0.65      0.62      0.63        39

    accuracy                           0.61       288
   macro avg       0.61      0.59      0.58       288
weighted avg       0.61      0.61      0.60       288



In [52]:
import joblib

joblib.dump(model, "emotion_model.pkl")
joblib.dump(encoder, "label_encoder.pkl")

print("Files created successfully")

Files created successfully
